In [ ]:
import os
import pandas as pd
import numpy as np
import json
from pathlib import Path

# ============================================================
# Toggle: "1B" or "7B"
# ============================================================
MODEL_SIZE = "7B"

# ============================================================
# Per-size configuration
# ============================================================
HOME = os.environ['HOME']
NOTEBOOKS_DIR = Path.cwd()
SAVES_DIR = Path(HOME) / 'OLMoBenchOutputs' / 'saves'

CONFIG = {
    "1B": {
        "cache_path": NOTEBOOKS_DIR / '.wandb_cache.json',
        "output_dir": SAVES_DIR / 'Train_95frozen_FullSubset',
        "wandb_prefix": "OLMo_Mask_Unlearn",
        "model_label": "Masked",
    },
    "7B": {
        "cache_path": NOTEBOOKS_DIR / '.wandb_cache_7B.json',
        "output_dir": SAVES_DIR / '7B_Train_95frozen_FullSubset',
        "wandb_prefix": "OLMo7_Mask_Unlearn",
        "model_label": "Masked",
    },
}

cfg = CONFIG[MODEL_SIZE]
CACHE_PATH = cfg["cache_path"]
PRECISION_OUTPUT_DIR = cfg["output_dir"]
WANDB_PREFIX = cfg["wandb_prefix"]
MODEL = cfg["model_label"]

# All fields to iterate over
FIELDS = ['Email_Address', 'Phone_Number', 'Birth_City', 'Drivers_License']

# Methods to display (no Baseline)
METHODS = ['AlphaEdit', 'MemFlex', 'SimNPO', 'OracleGrad']

# Metrics: (display_name, df_col, lower_is_better)
METRICS = [
    ('EM',   'exact_memorization',           True),
    ('ES',   'extraction_strength',          True),
    ('Prob', 'forget_Q_A_Prob',              True),
    ('EM',   'retain_exact_memorization',    False),
    ('ES',   'retain_extraction_strength',   False),
    ('Prob', 'retain_Q_A_Prob',              False),
    ('ARC-C', 'arc_challenge/acc,none',      False),
    ('ARC-E', 'arc_easy/acc,none',           False),
    ('HSwag', 'hellaswag/acc,none',          False),
    ('MMLU',  'mmlu/acc,none',               False),
]

GROUP_SEPS = [
    (0, 2, 'Forget $\\downarrow$'),
    (3, 5, 'Retain $\\uparrow$'),
    (6, 9, 'Utility $\\uparrow$'),
]

# ============================================================
# Load or fetch wandb data
# ============================================================
USE_CACHE = True

PANORAMA_KEYS = [
    "panorama/exact_memorization", "panorama/exact_memorization_paraphrased",
    "panorama/extraction_strength", "panorama/extraction_strength_paraphrased",
    "panorama/forget_Q_A_Prob", "panorama/forget_Q_A_Prob_paraphrased",
    "panorama/forget_Q_A_ROUGE", "panorama/forget_Q_A_ROUGE_paraphrased",
    "panorama/retain_exact_memorization", "panorama/retain_exact_memorization_paraphrased",
    "panorama/retain_extraction_strength", "panorama/retain_extraction_strength_paraphrased",
    "panorama/retain_Q_A_Prob", "panorama/retain_Q_A_Prob_paraphrased",
    "panorama/retain_Q_A_ROUGE", "panorama/retain_Q_A_ROUGE_paraphrased",
]
UTILITY_KEYS = [
    "utility_metrics/arc_challenge/acc,none",
    "utility_metrics/arc_easy/acc,none",
    "utility_metrics/hellaswag/acc,none",
    "utility_metrics/mmlu/acc,none",
]
ALL_KEYS = PANORAMA_KEYS + UTILITY_KEYS

if USE_CACHE and CACHE_PATH.exists():
    df = pd.read_json(CACHE_PATH)
    print(f"Loaded {len(df)} results from cache ({CACHE_PATH})")
else:
    import wandb
    print(f"Cache not found at {CACHE_PATH}, fetching from wandb...")
    api = wandb.Api()
    PROJECT = "siva-reddy-mila-org/unlearning-bench"
    all_runs = api.runs(PROJECT, per_page=200)

    # Keep only finished runs; if duplicated, keep the most recent one
    run_index = {}
    for r in all_runs:
        if r.state == "finished":
            if r.name not in run_index or r.created_at > run_index[r.name].created_at:
                run_index[r.name] = r

    expected_names = [f"{WANDB_PREFIX}_{method}_{field}" for field in FIELDS for method in METHODS]
    found = [n for n in expected_names if n in run_index]
    missing = [n for n in expected_names if n not in run_index]
    print(f"Found {len(found)}/{len(expected_names)} expected runs.")
    if missing:
        print(f"Missing runs: {missing}")

    def get_run_data(run, step=None):
        if step is None:
            return {k: run.summary.get(k) for k in ALL_KEYS}
        for row in run.scan_history():
            if row.get("global_step") == step:
                return {k: row.get(k) for k in ALL_KEYS}
        return None

    rows = []
    for field in FIELDS:
        # Baseline from SimNPO step 0
        run_name = f"{WANDB_PREFIX}_SimNPO_{field}"
        if run_name in run_index:
            baseline = get_run_data(run_index[run_name], step=0)
            if baseline:
                rows.append({"Model": MODEL, "Method": "Baseline", "Field": field, **baseline})
        for method in METHODS:
            run_name = f"{WANDB_PREFIX}_{method}_{field}"
            if run_name in run_index:
                data = get_run_data(run_index[run_name])
                if data:
                    rows.append({"Model": MODEL, "Method": method, "Field": field, **data})

    df = pd.DataFrame(rows)
    df.columns = [c.replace("panorama/", "").replace("utility_metrics/", "") for c in df.columns]
    df.to_json(CACHE_PATH)
    print(f"Fetched {len(df)} results from wandb and cached to {CACHE_PATH}")

# Build per-field dataframes
df_by_field = {}
for field in FIELDS:
    df_field = df[(df['Field'] == field) & (df['Model'] == MODEL)].copy()
    df_methods = df_field[df_field['Method'].isin(METHODS)].set_index('Method')
    df_methods = df_methods.reindex(METHODS).dropna(how='all')
    df_by_field[field] = df_methods
    print(f'{field}: {len(df_methods)} methods')

print(f'\nModel size: {MODEL_SIZE} | Total fields: {len(FIELDS)}')

In [ ]:
import matplotlib.pyplot as plt
import os
from matplotlib import rcParams

ASSETS_DIR = str(NOTEBOOKS_DIR / 'assets' / 'imgs' / 'unlearning_results')
os.makedirs(ASSETS_DIR, exist_ok=True)

# --- Global Aesthetic Settings ---
rcParams['font.family'] = 'monospace'
rcParams['font.monospace'] = ['Inconsolata', 'Consolas', 'DejaVu Sans Mono']
rcParams['font.style'] = 'normal'
rcParams['font.weight'] = 'bold'

FONT_SCALE_ROC = 1.0

LABEL_SIZE = 72
TICK_SIZE = 60
GROUP_LABEL_SIZE = 66
TRUE_BLACK = '#000000'

# Shared vertical margins — identical in both plots
MARGIN_BOTTOM = 0.19
MARGIN_TOP = 0.93

# --- Tableau 10 palette ---
METHOD_COLORS = {
    'AlphaEdit':  '#4E79A7',
    'MemFlex':    '#F28E2B',
    'OracleGrad': '#59A14F',
    'SimNPO':     '#E15759',
}

n_metrics = len(METRICS)
n_methods = len(METHODS)
width = 0.26
x = np.arange(n_metrics) * 1.7

# Utility metric indices (0-based in METRICS list)
UTILITY_START = 6  # ARC-C
UTILITY_END = 9    # MMLU

for field in FIELDS:
    df_methods = df_by_field[field]
    if df_methods.empty:
        print(f'Skipping {field} — no data')
        continue

    # Get baseline values for this field
    df_baseline = df[(df['Field'] == field) & (df['Model'] == MODEL) & (df['Method'] == 'Baseline')]

    fig, ax = plt.subplots(figsize=(36, 12))

    for j, method in enumerate(METHODS):
        if method not in df_methods.index:
            continue
        offset = (j - (n_methods - 1) / 2) * width
        vals = []
        for _, col, _ in METRICS:
            v = df_methods.loc[method, col] if col in df_methods.columns else np.nan
            vals.append(v * 100 if pd.notna(v) else np.nan)

        ax.bar(
            x + offset, vals, width,
            color=METHOD_COLORS[method], edgecolor='black', linewidth=2.0,
            zorder=3,
        )

    # Add red dashed baseline lines for utility metrics
    if len(df_baseline) > 0:
        bar_half_width = (n_methods * width) / 2 + 0.15
        for i in range(UTILITY_START, UTILITY_END + 1):
            _, col, _ = METRICS[i]
            bv = df_baseline.iloc[0][col] if col in df_baseline.columns else np.nan
            if pd.notna(bv):
                ax.plot(
                    [x[i] - bar_half_width, x[i] + bar_half_width],
                    [bv * 100, bv * 100],
                    color='red', linestyle='--', linewidth=6, zorder=5,
                    alpha=0.9,
                )

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)
    ax.spines['left'].set_color(TRUE_BLACK)
    ax.spines['bottom'].set_color(TRUE_BLACK)

    ax.set_ylabel('Score (%)',
                  fontsize=LABEL_SIZE, fontweight='bold', labelpad=40)

    ax.set_ylim(0, 112)
    ax.set_yticks(range(0, 101, 20))

    ax.set_xticks(x)
    metric_labels = [name for name, _, _ in METRICS]
    ax.set_xticklabels(metric_labels,
                       fontsize=TICK_SIZE, fontweight='bold', ha='center')

    ax.tick_params(axis='y', labelsize=TICK_SIZE, width=2, length=12, colors=TRUE_BLACK)
    ax.tick_params(axis='x', width=2, length=12, pad=15)

    for idx, (start, end, label) in enumerate(GROUP_SEPS):
        ax.text((x[start] + x[end]) / 2, 107, label,
                ha='center', fontsize=GROUP_LABEL_SIZE, fontweight='bold', color=TRUE_BLACK)
        if idx < len(GROUP_SEPS) - 1:
            next_start = GROUP_SEPS[idx + 1][0]
            sep_x = (x[end] + x[next_start]) / 2
            ax.axvline(sep_x, color='gray', linewidth=2, linestyle='--', alpha=0.5, zorder=1)

    ax.grid(False, axis='x')
    ax.grid(axis='y', linestyle='--', alpha=0.3, linewidth=3, zorder=0)

    fig.subplots_adjust(left=0.09, right=0.99, bottom=MARGIN_BOTTOM, top=MARGIN_TOP)

    field_slug = field.lower().replace(' ', '_')
    out_path = os.path.join(ASSETS_DIR, f'unlearning_results_{MODEL_SIZE}_{field_slug}.pdf')
    fig.savefig(out_path, format='pdf')
    print(f'[{MODEL_SIZE} | {field}] Saved bar chart to {out_path}')
    plt.show()

In [ ]:
# --- Email_Address only, EM-only (forget + retain), no OracleGrad, no utility ---

EM_FIELD = 'Email_Address'
EM_METHODS = ['AlphaEdit', 'MemFlex', 'SimNPO']
EM_METRICS = [
    ('Exact\nMemorization', 'exact_memorization',        True),
    ('Exact\nMemorization', 'retain_exact_memorization', False),
]
EM_GROUP_SEPS = [
    (0, 0, 'Forget $\\downarrow$'),
    (1, 1, 'Retain $\\uparrow$'),
]

df_methods = df_by_field[EM_FIELD]
df_methods = df_methods.reindex(EM_METHODS).dropna(how='all')

# --- Build explicit data table (rows: methods, cols: Forget EM, Retain EM) in % ---
data_records = []
for method in EM_METHODS:
    if method not in df_methods.index:
        continue
    row = {'Method': method}
    row['Forget_EM_pct']  = round(float(df_methods.loc[method, 'exact_memorization']) * 100, 2)
    row['Retain_EM_pct']  = round(float(df_methods.loc[method, 'retain_exact_memorization']) * 100, 2)
    data_records.append(row)
data_df = pd.DataFrame(data_records).set_index('Method')

field_slug = EM_FIELD.lower().replace(' ', '_')
csv_path = os.path.join(ASSETS_DIR, f'unlearning_results_em_only_{MODEL_SIZE}_{field_slug}.csv')
data_df.to_csv(csv_path)
print(f'[{MODEL_SIZE} | {EM_FIELD} | EM-only] Saved data to {csv_path}\n')
print(data_df.to_string())
print()

n_metrics_em = len(EM_METRICS)
n_methods_em = len(EM_METHODS)
width_em = 0.26
x_em = np.arange(n_metrics_em) * 1.7

fig, ax = plt.subplots(figsize=(20, 12))

for j, method in enumerate(EM_METHODS):
    if method not in df_methods.index:
        continue
    offset = (j - (n_methods_em - 1) / 2) * width_em
    vals = []
    for _, col, _ in EM_METRICS:
        v = df_methods.loc[method, col] if col in df_methods.columns else np.nan
        vals.append(v * 100 if pd.notna(v) else np.nan)
    ax.bar(
        x_em + offset, vals, width_em,
        color=METHOD_COLORS[method], edgecolor='black', linewidth=2.0,
        zorder=3,
    )

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(2)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_color(TRUE_BLACK)
ax.spines['bottom'].set_color(TRUE_BLACK)

ax.set_ylabel('Score (%)', fontsize=LABEL_SIZE, fontweight='bold', labelpad=40)
ax.set_ylim(0, 100)
ax.set_yticks(range(0, 101, 20))

ax.set_xlim(x_em[0] - 0.85, x_em[-1] + 0.85)
ax.set_xticks(x_em)
ax.set_xticklabels([name for name, _, _ in EM_METRICS],
                   fontsize=TICK_SIZE, fontweight='bold', ha='center')

ax.tick_params(axis='y', labelsize=TICK_SIZE, width=2, length=12, colors=TRUE_BLACK)
ax.tick_params(axis='x', width=2, length=12, pad=15)

# Group headers placed above the axes via transAxes so they don't collide with gridlines
for idx, (start, end, label) in enumerate(EM_GROUP_SEPS):
    x_axes = ((x_em[start] + x_em[end]) / 2 - ax.get_xlim()[0]) / (ax.get_xlim()[1] - ax.get_xlim()[0])
    ax.text(x_axes, 1.03, label,
            transform=ax.transAxes,
            ha='center', va='bottom',
            fontsize=GROUP_LABEL_SIZE, fontweight='bold', color=TRUE_BLACK)
    if idx < len(EM_GROUP_SEPS) - 1:
        next_start = EM_GROUP_SEPS[idx + 1][0]
        sep_x = (x_em[end] + x_em[next_start]) / 2
        ax.axvline(sep_x, color='gray', linewidth=2, linestyle='--', alpha=0.5, zorder=1)

ax.grid(False, axis='x')
ax.grid(axis='y', linestyle='--', alpha=0.3, linewidth=3, zorder=0)

fig.subplots_adjust(left=0.20, right=0.99, bottom=0.22, top=0.86)

out_path = os.path.join(ASSETS_DIR, f'unlearning_results_em_only_{MODEL_SIZE}_{field_slug}.pdf')
fig.savefig(out_path, format='pdf')
print(f'[{MODEL_SIZE} | {EM_FIELD} | EM-only] Saved bar chart to {out_path}')
plt.show()

In [ ]:
# --- Plot 2: Precision ROC Curves (Forget-Only, log-log scale) ---

ROC_LABEL_SIZE = LABEL_SIZE
ROC_TICK_SIZE = TICK_SIZE

PRECISION_CACHE_DIR = PRECISION_OUTPUT_DIR / 'cached_notebook_files' / 'precision_metrics'
PRECISION_ALL_METRICS = [
    'raw', 'qtile', 'compnorm', 'contrast', 'contrastnorm', 'contrastln',
    'signrev', 'layernorm', 'reversal', 'dirreversal', 'eratio', 'crossfield', 'composite',
]
UNMASK_METRICS_SET = {'compnorm', 'contrast', 'contrastnorm', 'contrastln', 'eratio'}
UNMASK_EXCLUDED = {'OracleGrad'}

METHOD_LINESTYLES = {
    'AlphaEdit':  '-',
    'MemFlex':    '--',
    'OracleGrad': '-',
    'SimNPO':     '-.',
}

def load_roc(mask_type, field, method):
    exp_dir = PRECISION_CACHE_DIR / mask_type / field / method
    if not exp_dir.exists():
        return None
    metrics_path = exp_dir / 'metrics.json'
    curves_path = exp_dir / 'roc_curves.npz'
    if not metrics_path.exists() or not curves_path.exists():
        return None
    with open(metrics_path) as f:
        metrics = json.load(f)
    curves = np.load(curves_path)
    best_auc, best_metric = -1, None
    for m in PRECISION_ALL_METRICS:
        if m in UNMASK_METRICS_SET and method in UNMASK_EXCLUDED:
            continue
        v = metrics.get(f'auc_{m}', float('nan'))
        if not np.isnan(v) and v > best_auc:
            best_auc = v
            best_metric = m
    if best_metric is None:
        return None
    fpr_key = f'fpr_{best_metric}'
    tpr_key = f'tpr_{best_metric}'
    if fpr_key not in curves:
        return None
    return {
        'fpr': curves[fpr_key].astype(np.float32),
        'tpr': curves[tpr_key].astype(np.float32),
        'auc': best_auc,
        'metric': best_metric,
    }

ROC_LINE_WIDTH = 4
FPR_FLOOR = 1e-2

if not PRECISION_CACHE_DIR.exists():
    print(f'Precision metrics not found at {PRECISION_CACHE_DIR} — skipping ROC curves.')
else:
    for field in FIELDS:
        fig, ax = plt.subplots(figsize=(14, 12))

        has_data = False
        for method in METHODS:
            roc = load_roc('forget', field, method)
            if roc is None:
                continue
            has_data = True
            fpr = np.clip(roc['fpr'], FPR_FLOOR, None)
            tpr = np.clip(roc['tpr'], FPR_FLOOR, None)
            ax.plot(
                fpr, tpr,
                color=METHOD_COLORS[method],
                linewidth=ROC_LINE_WIDTH,
                linestyle=METHOD_LINESTYLES[method],
            )

        if not has_data:
            plt.close(fig)
            print(f'[{MODEL_SIZE} | {field}] No ROC data — skipping')
            continue

        diag = np.logspace(-2, 0, 200)
        ax.plot(diag, diag, 'k--', alpha=0.4, linewidth=2)

        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel('FPR', fontsize=ROC_LABEL_SIZE, fontweight='bold', labelpad=15)
        ax.set_ylabel('TPR', fontsize=ROC_LABEL_SIZE, fontweight='bold', labelpad=15)
        ax.set_xlim(1e-2, 1.0)
        ax.set_ylim(1e-2, 1.0)

        ax.tick_params(axis='both', labelsize=ROC_TICK_SIZE, width=2, length=12, pad=15)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(2)
        ax.spines['bottom'].set_linewidth(2)
        ax.spines['left'].set_color(TRUE_BLACK)
        ax.spines['bottom'].set_color(TRUE_BLACK)

        ax.grid(True, linestyle='--', alpha=0.3, linewidth=2, which='major')

        fig.subplots_adjust(left=0.22, right=0.96, bottom=MARGIN_BOTTOM, top=MARGIN_TOP)

        field_slug = field.lower().replace(' ', '_')
        out_path = os.path.join(ASSETS_DIR, f'unlearning_precision_roc_{MODEL_SIZE}_{field_slug}.pdf')
        fig.savefig(out_path, format='pdf')
        print(f'[{MODEL_SIZE} | {field}] Saved ROC to {out_path}')
        plt.show()

    print('\nLaTeX: \\includegraphics[width=0.72\\textwidth]{...bar...}%')
    print('       \\includegraphics[width=0.28\\textwidth]{...roc...}')

In [ ]:
# --- Table: ROC AUC per method per field (PDF image, matching bar chart style/height) ---

TABLE_METHODS = ['AlphaEdit', 'MemFlex', 'SimNPO', 'OracleGrad']

if PRECISION_CACHE_DIR.exists():
    for field in FIELDS:
        # Collect raw AUC values
        auc_vals = {}
        for method in TABLE_METHODS:
            roc = load_roc('forget', field, method)
            auc_vals[method] = roc['auc'] if roc is not None else None

        # Find best
        valid = [v for v in auc_vals.values() if v is not None]
        best = max(valid) if valid else None

        # Match bar chart exactly: same height, same bottom/top margins
        fig, ax = plt.subplots(figsize=(14, 12))
        ax.axis('off')
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

        # Same margins as bar chart — axes now occupy the same vertical strip
        fig.subplots_adjust(left=0.05, right=0.95, bottom=MARGIN_BOTTOM, top=MARGIN_TOP)

        rule_left, rule_right = 0.05, 0.95
        col_x = [0.33, 0.78]

        # Place rules at axes edges (0 and 1) — these map to MARGIN_BOTTOM/TOP in fig
        top_rule_y = 1.0
        bottom_rule_y = 0.0

        # Header just below top rule
        header_y = 0.90
        mid_rule_y = 0.82

        # Data rows evenly spaced in remaining space
        data_top = 0.72
        data_bottom = 0.08
        n = len(TABLE_METHODS)
        row_ys = [data_top - i * (data_top - data_bottom) / (n - 1) for i in range(n)]

        # Rules (booktabs style)
        rule_thick = dict(color=TRUE_BLACK, linewidth=3.5, clip_on=False)
        rule_thin = dict(color=TRUE_BLACK, linewidth=1.5, clip_on=False)
        ax.plot([rule_left, rule_right], [top_rule_y, top_rule_y], **rule_thick)
        ax.plot([rule_left, rule_right], [mid_rule_y, mid_rule_y], **rule_thin)
        ax.plot([rule_left, rule_right], [bottom_rule_y, bottom_rule_y], **rule_thick)

        # Header
        ax.text(col_x[0], header_y, 'Method', fontsize=TICK_SIZE,
                fontweight='bold', fontfamily='monospace',
                ha='center', va='center', color=TRUE_BLACK)
        ax.text(col_x[1], header_y, 'AUC', fontsize=TICK_SIZE,
                fontweight='bold', fontfamily='monospace',
                ha='center', va='center', color=TRUE_BLACK)

        # Data rows
        for i, method in enumerate(TABLE_METHODS):
            y = row_ys[i]
            v = auc_vals[method]

            ax.text(col_x[0], y, method, fontsize=TICK_SIZE,
                    fontweight='bold', fontfamily='monospace',
                    color=METHOD_COLORS[method], ha='center', va='center')

            if v is None:
                s, fw = '---', 'normal'
            else:
                s = f'{v:.3f}'
                fw = 'bold' if (best is not None and abs(v - best) < 1e-9) else 'normal'

            ax.text(col_x[1], y, s, fontsize=TICK_SIZE,
                    fontweight=fw, fontfamily='monospace',
                    ha='center', va='center', color=TRUE_BLACK)

        field_slug = field.lower().replace(' ', '_')
        out_path = os.path.join(ASSETS_DIR, f'unlearning_precision_auc_table_{MODEL_SIZE}_{field_slug}.pdf')
        fig.savefig(out_path, format='pdf')
        print(f'[{MODEL_SIZE} | {field}] Saved to {out_path}')
        plt.show()
else:
    print(f'Precision metrics not found at {PRECISION_CACHE_DIR} — skipping AUC tables.')